# ISIC — captions only: Florence Attributes + FirstValid v3

Notebook riêng để xuất **captions.csv mới**, không train Florence hoặc mô hình segmentation.
Mặc định và bắt buộc `CAPTIONS_ONLY = True`.

1. Tái áp dụng policy **attributes-v2** lên câu dài đã được đăng ký từ run/bundle cũ.
2. Giữ nguyên mọi caption v2 đã đạt.
3. Với ảnh v2 bị mask: thử greedy dài hơn (128 token); nếu chưa đạt, thử 3 ứng viên từ beam search (beam=4, 192 token), lấy câu đạt đầu tiên.
4. Vẫn kiểm tra toàn bộ câu gốc theo blacklist chẩn đoán; vẫn yêu cầu 5–8 từ và nội dung thuộc tính. Không có ứng viên đạt thì giữ mask.

Mục tiêu là giảm mask so với v2 nhờ thêm phương án sinh, **không nới bộ lọc và không cam kết mask về 0**.
Caption tự sinh không phải ground truth. Ít mask hơn không chứng minh caption đúng hơn hoặc Dice cao hơn.
Không dùng GT mask/nhãn bệnh làm đầu vào và không chọn caption dựa trên metrics test.


## Inputs và cách chạy offline

Attach: **output/bundle ISIC M-TACS Florence cũ** (có captions.csv, status và generation protocol), **ảnh RGB ISIC gốc**, **Microsoft Florence-2-base** và **offline wheels Transformers 4.49** tương thích Python trên Kaggle.
Bundle đầy đủ có thể đã chứa Florence và wheels. Có thể dùng input `/kaggle/input/notebooks/nguyenducxu/notebook-mt-moi` như trước.
**Không cần DINO, best.pt segmentation hoặc GT masks.** CLIP chỉ cần nếu bật `ENCODE_CLIP_TEXT`.

Bật GPU, sửa đường dẫn ở cell dưới nếu auto-discovery tìm thấy nhiều nguồn, rồi **Run All**.
Nếu runtime dùng Transformers 5.x, captioner chạy subprocess 4.49 từ offline wheels; không sửa Transformers của kernel và không cần Internet.
Chạy lại cùng output để resume. Khi thay cấu hình hoặc nguồn CSV, dùng output mới.


In [ ]:
import os
# Set BEFORE importing transformers/tokenizers. No DataLoader/fork is used.
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

from pathlib import Path
import json
import sys

CAPTIONS_ONLY = True  # This notebook has no segmentation training cells.
if CAPTIONS_ONLY is not True:
    raise ValueError("Keep CAPTIONS_ONLY=True here; train M-TACS separately with the reviewed new CSV")

# Leave empty for discovery when exactly one different old CSV is attached.
# Example: /kaggle/input/datasets/YOUR_NAME/YOUR_BUNDLE/isic2017_mtacs_vitl16_448_web_bundle
# Accepts an old bundle, a saved run, shared/caption_generation, or its captions.csv.
PRIOR_CAPTIONS_SOURCE = os.getenv("ISIC_FIRSTVALID_PRIOR_SOURCE", "")
ISIC_ROOT = os.getenv("ISIC_ROOT", "")  # Folder containing the original ISIC RGB images.
FLORENCE_MODEL_REF = os.getenv("FLORENCE_MODEL_REF", "")  # Complete Microsoft Florence-2-base folder.
OFFLINE_ASSETS_ROOT = os.getenv("OFFLINE_ASSETS_ROOT", "/kaggle/input/notebooks/nguyenducxu/notebook-mt-moi")

ENCODE_CLIP_TEXT = False  # CSV-only by default; CLIP weights are optional.
CLIP_MODEL_LOCAL_PATH = os.getenv("CLIP_MODEL_LOCAL_PATH", "")

OUTPUT_DIR = Path(os.getenv("ISIC_FIRSTVALID_OUTPUT", "/kaggle/working/outputs/isic2017_captions_attributes_firstvalid_v3")).expanduser().resolve()
SEARCH_ROOTS = [Path("/kaggle/input")]
SETTINGS = {
    "seed": 42, "device": "cuda:0", "precision": "float32", "batch_size": 4,
    "max_new_tokens": 96,  # Original prefix setting retained as metadata, not a new initial pass.
    "retry_greedy_tokens": 128,
    "retry_beam_tokens": 192, "retry_beams": 4, "retry_returns": 3,
}
print("Caption-only:", CAPTIONS_ONLY, "| CLIP cache:", ENCODE_CLIP_TEXT)
print("New output:", OUTPUT_DIR)


In [ ]:
import importlib.util
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUNTIME_FILE = OUTPUT_DIR / "caption_firstvalid_runtime.py"
RUNTIME_FILE.write_text('from __future__ import annotations\nimport csv\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nimport os\nimport re\nimport shutil\nimport struct\nimport subprocess\nimport sys\nimport time\nfrom pathlib import Path\n\n\ndef atomic_csv(path, rows, columns):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    with temporary.open("w", encoding="utf-8", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=columns, extrasaction="ignore")\n        writer.writeheader()\n        writer.writerows(rows)\n        handle.flush()\n        os.fsync(handle.fileno())\n    temporary.replace(path)\n\ndef atomic_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + "\\n", encoding="utf-8")\n    temporary.replace(path)\n\ndef checkpoint_asset_identity(model_dir):\n    model_dir = Path(model_dir)\n    files = sorted(path for path in model_dir.iterdir() if path.is_file() and\n                   (path.suffix in {".py", ".json", ".safetensors", ".bin", ".txt"}))\n    if not any(path.suffix in {".safetensors", ".bin"} for path in files):\n        raise ValueError("Checkpoint has no weight files")\n    return {path.name: file_sha256(path) for path in files}\n\ndef file_sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for block in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\ndef object_hash(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=True,\n                                    allow_nan=False).encode()).hexdigest()\n\ndef read_csv_rows(path, required=()):\n    with Path(path).open(encoding="utf-8-sig", newline="") as handle:\n        reader = csv.DictReader(handle)\n        if not reader.fieldnames or len(set(reader.fieldnames)) != len(reader.fieldnames):\n            raise ValueError(f"Missing/duplicate CSV header: {path}")\n        if not set(required).issubset(reader.fieldnames):\n            raise ValueError(f"Missing required CSV columns: {sorted(set(required) - set(reader.fieldnames))}")\n        rows = list(reader)\n    if any(None in row or any(value is None for value in row.values()) for row in rows):\n        raise ValueError(f"Malformed/incomplete CSV row: {path}; original file was not modified")\n    return rows\n\ndef clean_dtd_caption(text):\n    import re\n    text = str(text).strip()\n    for _ in range(3):\n        before = text\n        text = re.sub(r"^(?:(?:the|this)\\s+)?(?:image|picture|photo(?:graph)?)\\s+"\n                      r"(?:is|shows|depicts|features|contains)\\s+", "", text, flags=re.I)\n        text = re.sub(r"^(?:(?:a|an|the)\\s+)?close[ -]up(?:\\s+view)?\\s+(?:of|showing)\\s+",\n                      "", text, flags=re.I)\n        text = re.sub(r"^(?:a|an|the)(?:\\s+|$)", "", text, flags=re.I)\n        if text == before:\n            break\n    words = re.findall(r"[A-Za-z0-9]+(?:[\'\\u2019][A-Za-z0-9]+)?", text)\n    selected = []\n    normalized_count = 0\n    for word in words:\n        count = len(re.findall(r"[A-Za-z0-9]+", word))\n        if normalized_count + count > 8:\n            break\n        selected.append(word)\n        normalized_count += count\n    return " ".join(selected)\n\ndef available_rows(directory, records, identity):\n    return [row for record in records if (row := cached_row(directory, record, identity)) is not None]\n\ndef cache_path(directory, image_id):\n    return Path(directory) / (object_hash(image_id) + ".json")\n\ndef cached_row(directory, record, identity):\n    path = cache_path(directory, record["image_id"])\n    if not path.is_file():\n        return None\n    result = json.loads(path.read_text(encoding="utf-8"))\n    if (result.get("identity") != identity or result.get("row", {}).get("image_id") != record["image_id"]\n            or result["row"].get("split") != record["split"]):\n        raise ValueError(f"Wrong cached image/protocol: {path}")\n    return result["row"]\n\ndef local_model_configs(roots):\n    seen = set()\n    for root in roots:\n        root = Path(root).expanduser()\n        if not root.is_dir():\n            continue\n        for current, dirs, files in os.walk(root):\n            dirs[:] = sorted(name for name in dirs if name not in {\n                "images", ".git", "__pycache__", "node_modules", ".cache"})\n            if len(Path(current).relative_to(root).parts) >= 9:\n                dirs[:] = []\n            if "config.json" not in files:\n                continue\n            path = Path(current).resolve()\n            if str(path) not in seen:\n                seen.add(str(path))\n                yield path\n\ndef lock_protocol(path, protocol):\n    path = Path(path)\n    if path.is_file():\n        if json.loads(path.read_text(encoding="utf-8")) != protocol:\n            raise ValueError(f"Changed images/model/prompt/runtime/settings. Use a new output directory: {path.parent}")\n    else:\n        if path.parent.exists() and any(path.parent.iterdir()):\n            raise ValueError(f"Refusing to adopt cached output without its protocol: {path.parent}")\n        atomic_json(path, protocol)\n    return object_hash(protocol)\n\ndef qwen_asset_identity(path):\n    result = checkpoint_asset_identity(path)\n    for template in sorted(Path(path).glob("*.jinja")):\n        result[template.name] = file_sha256(template)\n    return result\n\ndef resolve_clip_model(reference, roots):\n    def is_clip(path):\n        try:\n            return json.loads((path / "config.json").read_text(encoding="utf-8")).get("model_type") == "clip"\n        except (OSError, ValueError):\n            return False\n    if str(reference).strip() and Path(reference).is_dir():\n        path = Path(reference).resolve()\n        if not is_clip(path):\n            raise ValueError(f"Not a full local CLIP checkpoint: {path}")\n        return path\n    matches = [path for path in local_model_configs(roots) if is_clip(path)]\n    if len(matches) != 1:\n        raise FileNotFoundError("Set CLIP_MODEL_LOCAL_PATH to the exact full CLIP ViT-B/32 folder")\n    return matches[0]\n\ndef run_worker(worker_path, payload_path, phase, env, log_path):\n    Path(log_path).parent.mkdir(parents=True, exist_ok=True)\n    with Path(log_path).open("a", encoding="utf-8") as log:\n        process = subprocess.Popen([sys.executable, str(worker_path), str(payload_path), phase], env=env,\n            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace", bufsize=1)\n        try:\n            for line in process.stdout:\n                print(line, end="", flush=True)\n                log.write(line)\n                log.flush()\n            code = process.wait()\n        except BaseException:\n            process.terminate()\n            process.wait()\n            raise\n        finally:\n            process.stdout.close()\n    if code:\n        raise RuntimeError(f"{phase} worker exited with code {code}. See {log_path}; committed image caches are preserved")\n\ndef runtime_versions():\n    result = {}\n    for name in ("torch", "torchvision", "transformers", "accelerate", "tokenizers", "Pillow", "numpy"):\n        result[name] = importlib.metadata.version(name)\n    return result\n\ndef safetensors_parameter_count(files):\n    # Read headers only, not tensor contents; verify the attached model\'s actual scale.\n    count = 0\n    for path in files:\n        with Path(path).open("rb") as handle:\n            raw_length = handle.read(8)\n            if len(raw_length) != 8:\n                raise ValueError(f"Truncated safetensors header: {path}")\n            length = struct.unpack("<Q", raw_length)[0]\n            if not 2 <= length <= 64 * 1024 * 1024 or length + 8 > Path(path).stat().st_size:\n                raise ValueError(f"Invalid safetensors header length: {path}")\n            header = json.loads(handle.read(length))\n        for name, tensor in header.items():\n            if name == "__metadata__":\n                continue\n            if tensor.get("dtype") not in {"F16", "BF16", "F32"}:\n                raise ValueError("Use original floating-point weights, not quantized/FP8 weights, for this comparison")\n            shape = tensor.get("shape", [])\n            if not all(type(size) is int and size >= 0 for size in shape):\n                raise ValueError(f"Invalid tensor shape: {path}: {name}")\n            count += math.prod(shape)\n    return count\n\ndef weight_files(path):\n    path = Path(path).resolve()\n    index = path / "model.safetensors.index.json"\n    if index.is_file():\n        names = set(json.loads(index.read_text(encoding="utf-8")).get("weight_map", {}).values())\n        if not names:\n            raise ValueError(f"Empty safetensors weight index: {index}")\n        if any(not isinstance(name, str) or "/" in name or "\\\\" in name or Path(name).is_absolute() for name in names):\n            raise ValueError(f"Weight index must reference shard filenames inside its model folder: {index}")\n        files = [path / name for name in sorted(names)]\n    else:\n        files = [path / "model.safetensors"]\n    for file in files:\n        # HF snapshot files may be read-only symlinks into a sibling blobs directory.\n        if not file.is_file() or file.suffix != ".safetensors":\n            raise FileNotFoundError(f"Missing/unsafe local safetensors shard: {file}")\n    return files\n\nPOLICY = \'isic_visible_attributes_v2\'\n\nCLINICAL_PATTERN = r"\\b(?:melanoma|melanomas|melanocytic|nevus|nevi|naevus|naevi|benign|malignant|malignancy|cancer|cancerous|carcinoma|keratosis|keratoses|diagnosis|diagnosed)\\b"\n\nCOLUMNS = ["image_id", "split", "raw_caption", "final_caption", "caption_valid", "leakage_detected",\n           "clinical_term_detected", "word_count", "source_long_caption", "generation_token_limit_reached",\n           "vlm_model", "prompt_version", "caption_input_policy", "captioner_fingerprint", "retry_count"]\n\ndef caption_policy(response, token_limit=False):\n    caption = clean_dtd_caption(response)\n    count = len(re.findall(r"[A-Za-z0-9]+", caption))\n    clinical = bool(re.search(CLINICAL_PATTERN, str(response), re.I))\n    valid = 5 <= count <= 8 and not clinical and not token_limit\n    return caption, valid, clinical, count\n\ndef validate_caption_records(records):\n    if not records or len({row["image_id"] for row in records}) != len(records):\n        raise ValueError("Caption image IDs must be nonempty and unique across splits")\n    for row in records:\n        if set(row) != {"image_id", "image_path", "image_sha256", "split"}:\n            raise ValueError("Caption worker accepts pixels/identity only, never masks, class labels or clinical metadata")\n        if row["split"] not in {"train", "val", "test"}:\n            raise ValueError("Unexpected split")\n        if file_sha256(row["image_path"]) != row["image_sha256"]:\n            raise ValueError(f"Image changed: {row[\'image_id\']}")\n\ndef validate_text_cache(cache, rows, csv_sha256, identity):\n    import torch\n    expected_valid = torch.tensor([row["caption_valid"].lower() == "true" for row in rows], dtype=torch.bool)\n    vectors, valid = cache["embeddings"], cache["valid"]\n    if (cache["identity"] != identity or cache["image_ids"] != [row["image_id"] for row in rows]\n            or cache["caption_csv_sha256"] != csv_sha256 or vectors.ndim != 2 or len(vectors) != len(rows)\n            or valid.dtype != torch.bool or not torch.equal(valid.cpu(), expected_valid)):\n        raise ValueError("Wrong CLIP text cache identity/IDs/shape/masks")\n    if not torch.isfinite(vectors).all() or torch.count_nonzero(vectors[~valid]):\n        raise ValueError("Invalid captions must have finite zero embeddings")\n    if valid.any() and not torch.allclose(vectors[valid].float().norm(dim=1), torch.ones(int(valid.sum())), atol=1e-4):\n        raise ValueError("Valid CLIP caption vectors must have unit norm")\n\nMETHOD = \'florence_attributes_firstvalid_v3\'\n\nMODEL_ID = "microsoft/Florence-2-base"\n\nTASK_PROMPT = "<MORE_DETAILED_CAPTION>"\n\ndef validate_florence_prefix_folder(path):\n    path = Path(path).expanduser().resolve()\n    config = json.loads((path / "config.json").read_text(encoding="utf-8"))\n    if (config.get("model_type") != "florence2"\n            or "AutoModelForCausalLM" not in config.get("auto_map", {})):\n        raise ValueError("Florence-Prefix requires the original legacy Microsoft Florence-2-base checkpoint")\n    required = ("configuration_florence2.py", "modeling_florence2.py", "processing_florence2.py",\n                "preprocessor_config.json", "tokenizer_config.json")\n    if any(not (path / name).is_file() for name in required):\n        raise FileNotFoundError("Incomplete local Florence model code/processor/tokenizer files")\n    if not ((path / "tokenizer.json").is_file() or all((path / name).is_file() for name in ("vocab.json", "merges.txt"))):\n        raise FileNotFoundError("Missing Florence tokenizer vocabulary")\n    if (path / "adapter_config.json").exists() or "-ft" in path.name.lower():\n        raise ValueError("Use the original frozen base model, not a LoRA or -ft captioner")\n    count = safetensors_parameter_count(weight_files(path))\n    if not 180_000_000 <= count <= 300_000_000:\n        raise ValueError(f"Expected Florence-2-base scale, not large/quantized/adapter weights: {count:,}")\n    return path, count\n\nclass IsicFlorencePrefixCaptioner:\n    def __init__(self, path, settings):\n        import torch\n        import transformers\n        from transformers import AutoModelForCausalLM, AutoProcessor\n\n        if not transformers.__version__.startswith("4.49."):\n            raise RuntimeError("Run the legacy Florence worker with the isolated offline Transformers 4.49 environment")\n        self.device = torch.device(settings.get("device", "cuda:0"))\n        self.settings = settings\n        self.dtype = torch.float32\n        self.processor = AutoProcessor.from_pretrained(str(path), trust_remote_code=True, local_files_only=True)\n        model, info = AutoModelForCausalLM.from_pretrained(\n            str(path), torch_dtype=self.dtype, trust_remote_code=True, local_files_only=True,\n            attn_implementation="eager", low_cpu_mem_usage=True, output_loading_info=True,\n        )\n        missing = [key for key in info.get("missing_keys", []) if not key.endswith("position_ids")]\n        if missing or info.get("mismatched_keys"):\n            raise ValueError(f"Incomplete Florence weights: {missing}")\n        self.model = model.to(self.device).eval().requires_grad_(False)\n\n    def generate(self, images):\n        import torch\n\n        inputs = self.processor(text=[TASK_PROMPT] * len(images), images=images,\n                                return_tensors="pt", padding=True)\n        moved = {key: value.to(device=self.device, dtype=self.dtype if key == "pixel_values" else value.dtype)\n                 for key, value in inputs.items() if torch.is_tensor(value)}\n        with torch.inference_mode():\n            output = self.model.generate(**moved, max_new_tokens=self.settings["max_new_tokens"],\n                                         num_beams=1, do_sample=False, use_cache=True, early_stopping=False)\n        texts = self.processor.batch_decode(output, skip_special_tokens=False)\n        if len(texts) != len(images):\n            raise ValueError("Florence response count mismatch")\n        eos = self.model.generation_config.eos_token_id\n        eos = set(eos if isinstance(eos, (list, tuple)) else [eos])\n        responses = []\n        for text, image, sequence in zip(texts, images, output.cpu().tolist()):\n            parsed = self.processor.post_process_generation(text, task=TASK_PROMPT,\n                                                           image_size=(image.width, image.height))\n            if not isinstance(parsed, dict) or not isinstance(parsed.get(TASK_PROMPT), str):\n                raise ValueError("Florence did not return the requested detailed-caption field")\n            # Florence is encoder-decoder: its first decoder token can itself be EOS.\n            new_tokens = sequence[1:]\n            limited = (len(new_tokens) >= self.settings["max_new_tokens"]\n                       and not any(token in eos for token in new_tokens))\n            responses.append((parsed[TASK_PROMPT].strip(), limited))\n        return responses\n\nMODEL_IDS = {METHOD: MODEL_ID}\nCAPTION_PROMPT = TASK_PROMPT\n\ndef make_generation_row(record, response, token_limit, identity):\n    caption, valid, clinical, count = caption_policy(response, token_limit)\n    return {"image_id": record["image_id"], "split": record["split"], "raw_caption": caption,\n            "final_caption": caption, "caption_valid": valid, "leakage_detected": clinical,\n            "clinical_term_detected": clinical, "word_count": count, "source_long_caption": response,\n            "generation_token_limit_reached": token_limit, "vlm_model": MODEL_IDS[METHOD],\n            "prompt_version": POLICY, "caption_input_policy": POLICY,\n            "captioner_fingerprint": identity, "retry_count": 0}\n\ndef generate_prefix_records(payload, identity, factory=IsicFlorencePrefixCaptioner):\n    import torch\n    from PIL import Image, ImageOps\n    from tqdm.auto import tqdm\n    records, settings = payload["records"], payload["settings"]\n    directory = Path(payload["output"])\n    cache = directory / "generated"\n    todo = [row for row in records if cached_row(cache, row, identity) is None]\n    if payload.get("max_new_images", 0):\n        todo = todo[:payload["max_new_images"]]\n    print(f"ISIC captions: cached={len(records) - sum(cached_row(cache, r, identity) is None for r in records)}, queued={len(todo)}", flush=True)\n    if todo:\n        captioner = factory(payload["model_path"], settings)\n        offset, batch_size = 0, settings["batch_size"]\n        progress = tqdm(total=len(todo), desc="ISIC Florence-Prefix", unit="image")\n        try:\n            while offset < len(todo):\n                group = todo[offset:offset + batch_size]\n                images = []\n                try:\n                    for record in group:\n                        if file_sha256(record["image_path"]) != record["image_sha256"]:\n                            raise ValueError("Image changed after hashing")\n                        with Image.open(record["image_path"]) as source:\n                            images.append(ImageOps.exif_transpose(source).convert("RGB"))\n                    responses = captioner.generate(images)\n                    if len(responses) != len(group):\n                        raise ValueError("Caption batch is incomplete")\n                except torch.cuda.OutOfMemoryError:\n                    if batch_size == 1:\n                        raise RuntimeError("Florence OOM at batch 1; free GPU memory or use a larger GPU. Committed captions are resumable") from None\n                    batch_size = max(1, batch_size // 2)\n                    gc.collect()\n                    torch.cuda.empty_cache()\n                    continue\n                finally:\n                    for image in images:\n                        image.close()\n                for record, (response, limited) in zip(group, responses):\n                    row = make_generation_row(record, response, limited, identity)\n                    atomic_json(cache_path(cache, record["image_id"]), {"identity": identity, "row": row})\n                offset += len(group)\n                progress.update(len(group))\n        finally:\n            progress.close()\n            del captioner\n            gc.collect()\n            torch.cuda.empty_cache()\n    rows = available_rows(cache, records, identity)\n    complete = len(rows) == len(records)\n    path = directory / ("captions.csv" if complete else "captions_partial.csv")\n    atomic_csv(path, rows, COLUMNS)\n    status = {"complete": complete, "rows": len(rows), "expected": len(records), "identity": identity,\n              "csv_path": str(path.resolve()), "csv_sha256": file_sha256(path),\n              "valid_by_split": {role: sum(bool(row["caption_valid"]) for row in rows if row["split"] == role)\n                                 for role in ("train", "val", "test")}}\n    atomic_json(directory / "status.json", status)\n    print(status, flush=True)\n    return status\n\ndef validated_export(payload):\n    directory = Path(payload["output"])\n    status = json.loads((directory / "status.json").read_text(encoding="utf-8"))\n    if not status["complete"]:\n        raise RuntimeError("Caption smoke test/partial output cannot train segmentation; set MAX_NEW_CAPTION_IMAGES=0 and resume")\n    path = directory / "captions.csv"\n    if file_sha256(path) != status["csv_sha256"]:\n        raise ValueError("Caption CSV changed after generation")\n    rows = read_csv_rows(path, COLUMNS)\n    expected = [(r["image_id"], r["split"]) for r in payload["records"]]\n    if [(r["image_id"], r["split"]) for r in rows] != expected:\n        raise ValueError("Wrong/missing/duplicate image IDs or split order in captions.csv")\n    for row in rows:\n        clean, valid, clinical, count = caption_policy(row["source_long_caption"], str(row["generation_token_limit_reached"]).lower() == "true")\n        if row["final_caption"] != clean or row["caption_valid"].lower() != str(valid).lower() or int(row["word_count"]) != count:\n            raise ValueError("Caption policy fields do not match their original Florence response")\n        if row["captioner_fingerprint"] != status["identity"] or row["caption_input_policy"] != POLICY or row["vlm_model"] != MODEL_IDS[METHOD]:\n            raise ValueError("Wrong caption generation provenance")\n    if not any(row["split"] == "train" and row["caption_valid"].lower() == "true" for row in rows):\n        raise ValueError("No usable training captions; inspect CSV and policy, do not force masks/gates on")\n    return rows, status\n\ndef encode_isic_texts(payload):\n    import torch\n    from transformers import AutoTokenizer, CLIPTextModelWithProjection\n    rows, status = validated_export(payload)\n    output = Path(payload["output"])\n    protocol = {"schema": 1, "caption_csv_sha256": status["csv_sha256"],\n                "clip_assets": qwen_asset_identity(payload["clip_path"]), "policy": POLICY,\n                "runtime": runtime_versions(), "code_sha256": payload["code_sha256"]}\n    identity = lock_protocol(output / "text_cache" / "protocol.json", protocol)\n    target = output / "text_cache" / "clip_text.pt"\n    if target.is_file():\n        cached = torch.load(target, map_location="cpu", weights_only=True)\n        validate_text_cache(cached, rows, status["csv_sha256"], identity)\n        atomic_json(output / "text_cache" / "status.json", {"identity": identity, "sha256": file_sha256(target),\n                    "valid": int(cached["valid"].sum()), "total": len(rows), "shape": list(cached["embeddings"].shape)})\n        print("Reusing CLIP text cache:", target, flush=True)\n        return\n    tokenizer = AutoTokenizer.from_pretrained(payload["clip_path"], local_files_only=True)\n    model, info = CLIPTextModelWithProjection.from_pretrained(payload["clip_path"], local_files_only=True,\n                                                            output_loading_info=True)\n    missing = [key for key in info.get("missing_keys", []) if not key.endswith("position_ids")]\n    if missing or info.get("mismatched_keys"):\n        raise ValueError(f"Incomplete CLIP text weights: {missing}")\n    model = model.to("cuda:0").eval().requires_grad_(False)\n    valid = torch.tensor([row["caption_valid"].lower() == "true" for row in rows], dtype=torch.bool)\n    vectors = torch.zeros((len(rows), model.config.projection_dim), dtype=torch.float32)\n    positions = valid.nonzero().flatten().tolist()\n    for start in range(0, len(positions), 128):\n        group = positions[start:start + 128]\n        tokens = tokenizer([rows[i]["final_caption"] for i in group], padding=True, truncation=True,\n                           max_length=model.config.max_position_embeddings, return_tensors="pt")\n        tokens = {key: value.to("cuda:0") for key, value in tokens.items()}\n        with torch.inference_mode():\n            values = torch.nn.functional.normalize(model(**tokens).text_embeds.float(), dim=1)\n        vectors[group] = values.cpu()\n    temporary = target.with_suffix(".pt.tmp")\n    torch.save({"identity": identity, "image_ids": [row["image_id"] for row in rows],\n                "embeddings": vectors, "valid": valid, "caption_csv_sha256": status["csv_sha256"]}, temporary)\n    temporary.replace(target)\n    atomic_json(output / "text_cache" / "status.json", {"identity": identity, "sha256": file_sha256(target),\n                "valid": int(valid.sum()), "total": len(rows), "shape": list(vectors.shape)})\n    print("CLIP vectors:", tuple(vectors.shape), "valid:", int(valid.sum()), flush=True)\n\ndef florence_checkpoint(model_ref, roots):\n    def checked(path):\n        path = Path(path)\n        try:\n            config = json.loads((path / "config.json").read_text(encoding="utf-8"))\n        except (OSError, ValueError):\n            return None\n        if config.get("model_type") != "florence2":\n            return None\n        indexes = list(path.glob("*.index.json"))\n        weights = any((path / name).is_file() for name in ("model.safetensors", "pytorch_model.bin"))\n        for index in indexes:\n            try:\n                shards = set(json.loads(index.read_text(encoding="utf-8")).get("weight_map", {}).values())\n                weights |= bool(shards) and all((path / shard).is_file() for shard in shards)\n            except (OSError, ValueError):\n                continue\n        if not weights:\n            return None\n        backend = "legacy" if "AutoModelForCausalLM" in config.get("auto_map", {}) else "native"\n        if backend == "legacy" and not all((path / name).is_file() for name in (\n                "configuration_florence2.py", "modeling_florence2.py", "processing_florence2.py")):\n            return None\n        return path.resolve(), backend\n\n    requested = Path(model_ref).expanduser()\n    if requested.is_dir():\n        result = checked(requested)\n        if result:\n            return result\n        for config in sorted(requested.rglob("config.json")):\n            result = checked(config.parent)\n            if result:\n                return result\n        raise FileNotFoundError(f"Florence folder is incomplete: {requested}; need config, weights, processor/tokenizer and local model code.")\n    for root in roots:\n        root = Path(root)\n        if not root.is_dir():\n            continue\n        for current, dirs, files in os.walk(root):\n            dirs[:] = sorted(name for name in dirs if name not in {"images", "__pycache__", ".git"})\n            if len(Path(current).relative_to(root).parts) >= 8:\n                dirs[:] = []\n            if "config.json" in files:\n                result = checked(current)\n                if result:\n                    return result\n    raise FileNotFoundError(\n        f"No complete local Florence-2 checkpoint found for {model_ref}. "\n        "Attach notebook-mt-moi/Florence-2-base or set FLORENCE_MODEL_REF to the correct local folder. No download is attempted."\n    )\n\ndef offline_caption_wheels(roots, destination):\n    import zipfile\n    destination = Path(destination)\n    wheel_dirs = set()\n    prefixes = ("transformers-", "tokenizers-", "huggingface_hub-", "accelerate-", "einops-", "timm-")\n    for root in roots:\n        root = Path(root)\n        if not root.is_dir():\n            continue\n        for current, dirs, files in os.walk(root):\n            dirs[:] = sorted(name for name in dirs if name not in {"images", "__pycache__", ".git", "Florence-2-base"})\n            if len(Path(current).relative_to(root).parts) >= 7:\n                dirs[:] = []\n            for name in sorted(files):\n                path = Path(current) / name\n                if name.endswith(".whl"):\n                    wheel_dirs.add(str(path.parent))\n                elif name.lower() in {"offline_wheels.zip", "wheels.zip"}:\n                    with zipfile.ZipFile(path) as archive:\n                        for member in archive.infolist():\n                            filename = Path(member.filename.replace("\\\\", "/")).name\n                            if not filename.endswith(".whl") or not filename.startswith(prefixes):\n                                continue\n                            destination.mkdir(parents=True, exist_ok=True)\n                            target = destination / filename\n                            if not target.is_file() or target.stat().st_size != member.file_size:\n                                with archive.open(member) as source, target.open("wb") as handle:\n                                    shutil.copyfileobj(source, handle)\n                            wheel_dirs.add(str(destination))\n    return sorted(wheel_dirs)\n\ndef caption_runtime_environment(backend, asset_root, runtime_dir):\n    import subprocess\n    import sys\n    runtime_dir = Path(runtime_dir)\n    runtime_dir.mkdir(parents=True, exist_ok=True)\n    env = dict(os.environ)\n    env.update(TOKENIZERS_PARALLELISM="false", HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1",\n               HF_MODULES_CACHE=str(runtime_dir / "hf_modules"), PYTHONUNBUFFERED="1")\n    target = runtime_dir / "legacy_packages"\n    probe = (\n        "import transformers; "\n        + ("assert transformers.__version__.startswith(\'4.49.\'), transformers.__version__; " if backend == "legacy" else "")\n        + "from transformers import AutoProcessor, AutoTokenizer, CLIPModel; "\n        + ("from transformers import AutoModelForCausalLM; " if backend == "legacy" else\n           "from transformers import Florence2ForConditionalGeneration; ")\n        + "import einops, timm, accelerate; print(transformers.__version__)"\n    )\n    if target.is_dir():\n        env["PYTHONPATH"] = str(target) + (os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")\n    result = subprocess.run([sys.executable, "-c", probe], env=env, capture_output=True, text=True)\n    if result.returncode == 0:\n        print("Caption subprocess Transformers:", result.stdout.strip())\n        return env\n    if backend != "legacy":\n        raise RuntimeError("The local native Florence checkpoint cannot load in this runtime. "\n                           "Use a matching native Transformers installation or the attached legacy model + offline 4.49 wheels.\\n"\n                           + result.stderr[-3000:])\n    wheel_dirs = offline_caption_wheels([Path(asset_root), Path("/kaggle/input")], runtime_dir / "wheels")\n    if not wheel_dirs:\n        raise FileNotFoundError(\n            "The attached Microsoft Florence checkpoint needs Transformers 4.49.x. "\n            f"No offline wheels found under {asset_root}. Attach offline_wheels.zip/wheels.zip or a wheel folder "\n            "with transformers==4.49.0, tokenizers 0.21.x and huggingface-hub <1 compatible with this Python version. "\n            "The notebook never enables Internet or modifies the kernel\'s Transformers."\n        )\n    command = [sys.executable, "-m", "pip", "install", "--no-index", "--only-binary=:all:",\n               "--no-deps", "--upgrade", "--target", str(target), "transformers==4.49.0",\n               "tokenizers>=0.21,<0.22", "huggingface-hub>=0.26,<1"]\n    import importlib.util\n    for module in ("einops", "timm", "accelerate"):\n        if importlib.util.find_spec(module) is None:\n            command.append(module)\n    for directory in wheel_dirs:\n        command.extend(["--find-links", directory])\n    print("Preparing isolated Florence dependencies from attached offline wheels; the notebook kernel is unchanged.")\n    installed = subprocess.run(command, env=env, capture_output=True, text=True)\n    if installed.returncode:\n        raise RuntimeError("Offline Florence dependency setup failed. Wheels must match Kaggle\'s Python/platform.\\n"\n                           + installed.stdout[-2000:] + installed.stderr[-3000:])\n    base_pythonpath = os.environ.get("PYTHONPATH", "")\n    env["PYTHONPATH"] = str(target) + (os.pathsep + base_pythonpath if base_pythonpath else "")\n    checked = subprocess.run([sys.executable, "-c", probe], env=env, capture_output=True, text=True)\n    if checked.returncode:\n        raise RuntimeError("The offline Florence environment still lacks a compatible dependency. "\n                           "No kernel packages were changed.\\n" + checked.stderr[-4000:])\n    print("Isolated caption subprocess Transformers:", checked.stdout.strip())\n    return env\n\ndef florence_worker_main(request_path, phase, *, generate_records, encode_texts):\n    import torch\n\n    payload = json.loads(Path(request_path).read_text(encoding="utf-8"))\n    validate_caption_records(payload["records"])\n    if not torch.cuda.is_available():\n        raise RuntimeError("Enable a CUDA GPU")\n    settings = payload["settings"]\n    if type(payload.get("max_new_images", 0)) is not int or payload.get("max_new_images", 0) < 0:\n        raise ValueError("max_new_images must be a nonnegative integer")\n    for key in ("batch_size", "max_new_tokens"):\n        if type(settings[key]) is not int or settings[key] < 1:\n            raise ValueError(f"{key} must be a positive integer")\n    if type(settings["seed"]) is not int or not 0 <= settings["seed"] < 2**32:\n        raise ValueError("Caption seed must be an integer in [0, 2**32)")\n    torch.manual_seed(settings["seed"])\n    torch.cuda.manual_seed_all(settings["seed"])\n    if phase == "generate":\n        from importlib.metadata import version\n        if not version("transformers").startswith("4.49."):\n            raise RuntimeError("Legacy Florence generation must use the isolated Transformers 4.49 runtime")\n        path, count = validate_florence_prefix_folder(payload["model_path"])\n        protocol = {"schema": 1, "method": METHOD, "model_id": MODEL_ID,\n                    "model_assets": checkpoint_asset_identity(path), "parameters": count,\n                    "prompt": TASK_PROMPT, "decoding": "greedy_beam1_extract_visible_attribute_phrase",\n                    "dtype": "float32",\n                    "policy": \'isic_visible_attributes_v2\',\n                    "images": [{key: row[key] for key in ("image_id", "split", "image_sha256")}\n                               for row in payload["records"]],\n                    "settings": settings, "runtime": runtime_versions(), "code_sha256": payload["code_sha256"],\n                    "long_response_source": prior_caption_registration(payload)[0]}\n        output = Path(payload["output"])\n        identity = lock_protocol(output / "generation_protocol" / "protocol.json", protocol)\n        if not (output / "status.json").exists() and (output / "captions.csv").exists():\n            raise ValueError("Unregistered captions.csv; use a fresh output root")\n        generate_records(payload, identity)\n    elif phase == "encode":\n        encode_texts(payload)\n    else:\n        raise ValueError("Phase must be generate or encode")\n\n\ndef attribute_caption_details(response):\n    import re\n    colors = set("brown black pink pinkish red reddish purple purplish blue bluish yellow yellowish white whitish beige tan gray grey orange multicolored".split())\n    shapes = set("round rounded circular oval irregular irregularly shaped asymmetric asymmetrical elongated angular curved patch patches spot spots circle circles blob blobs".split())\n    boundaries = set("border borders edge edges outline outlines boundary boundaries rim perimeter jagged blurred defined sharp distinct".split())\n    textures = set("rough smooth flaky dry raised flat wrinkled granular speckled dotted dots speckles lines network reticular scaly cracked bumpy fuzzy hairy hairs streak streaks".split())\n    sizes = set("small tiny large little thin thick narrow wide central center centre scattered clustered surrounding dark darker light lighter pale uneven".split())\n    stop = set("a an the of to in on at from for with and or but is are was were be been being that which this these those it its their there as than by has have appears appear seems seem looks look very slightly covered larger smaller".split())\n    filler = re.compile(r"\\b(?:person[\'’]s|person|image|photo|picture|specifically|appears|seems|depicts|shows|petri|dish|container|bowl|substance|object)\\b", re.I)\n    candidates = []\n    seen = set()\n    for sentence in re.split(r"[.!?;\\n]+", str(response)):\n        sentence = sentence.strip().lower()\n        sentence = re.sub(r"^(?:in|at)\\s+(?:the\\s+)?cent(?:er|re)\\s+of\\s+(?:the\\s+)?(?:skin|image|photo|picture)\\s*,?\\s*(?:there\\s+(?:is|are)\\s+)?", "central ", sentence)\n        sentence = re.sub(r"^(?:(?:the|this)\\s+)?(?:image|photo|picture)\\s+(?:shows|is|depicts|features)\\s+", "", sentence)\n        sentence = re.sub(r"^(?:a\\s+)?close[ -]up(?:\\s+view)?\\s+of\\s+", "", sentence)\n        sentence = re.sub(r"^(?:the\\s+)?skin\\s+(?:appears|seems)\\s+to\\s+be\\s+", "", sentence)\n        sentence = re.sub(r"^(?:the\\s+)?(?:substance|surface|area)\\s+is\\s+", "", sentence)\n        sentence = re.sub(r"\\b(?:a|an|the)\\b", "", sentence)\n        sentence = re.sub(r"\\bin (?:the )?colou?r\\b", "", sentence)\n        sentence = re.sub(r"\\b(?:that|which) (?:is|are)\\b", "", sentence)\n        sentence = re.sub(r"\\band (?:has|have)(?: a)?\\b", "and", sentence)\n        words = re.findall(r"[a-z0-9]+(?:[\'’][a-z0-9]+)?", sentence)\n        # A compact noun phrase may move attributes before their own source\n        # noun. Never borrow a colour/texture from another sentence/referent.\n        noun = next((w for w in words if w in {"area", "patch", "spot", "circle", "surface"}), None)\n        if noun:\n            noun_colors = [w for w in words if w in colors]\n            shape_adjectives = [w for w in words if w in {"round", "rounded", "circular", "oval", "irregular", "asymmetric", "elongated"}]\n            size = next((w for w in words if w in {"small", "tiny", "large", "thin", "thick"}), None)\n            tone = next((w for w in words if w in {"dark", "light", "pale"}), None)\n            composed = [w for w in [size, "central" if "central" in words else None, tone,\n                        noun_colors[0] if noun_colors else None,\n                        shape_adjectives[0] if shape_adjectives else None, noun] if w]\n            unambiguous = (len(set(noun_colors)) == 1\n                and sum(w in {"area", "patch", "spot", "circle", "surface"} for w in words) == 1\n                and not set(words) & {"background", "next", "beside", "near", "surrounding", "around", "than", "while"})\n            if unambiguous and len(composed) == len(set(composed)) and 5 <= len(composed) <= 8 and noun_colors and (shape_adjectives or noun in shapes):\n                caption = " ".join(composed)\n                tokens = set(composed)\n                groups = [tokens & g for g in (colors, shapes, boundaries, textures, sizes)]\n                info = set.union(*groups)\n                score = (3 * len(groups[0]) + 6 * len(groups[1]) + 5 * len(groups[2]) + 2 * len(groups[3])\n                         + .5 * len(groups[4]) + 2 * len(info) / len(composed) + 1.)\n                if caption not in seen:\n                    candidates.append({"caption": caption, "score": round(score, 8), "attributes": sorted(info)})\n                    seen.add(caption)\n        for start in range(len(words)):\n            if words[start] in stop:\n                continue\n            for width in range(5, 9):\n                selected = words[start:start + width]\n                if len(selected) != width or selected[-1] in stop:\n                    continue\n                caption = " ".join(selected)\n                if re.search(r"\\b(?:there is|there are|it is|appears to|skin there)\\b", caption):\n                    continue\n                if caption in seen or not 5 <= len(re.findall(r"[A-Za-z0-9]+", caption)) <= 8 or filler.search(caption):\n                    continue\n                if selected[-1] in colors | sizes | textures:\n                    continue  # Do not cut immediately before the described noun.\n                tokens = set(selected)\n                groups = [tokens & group for group in (colors, shapes, boundaries, textures, sizes)]\n                # Generic skin/texture-only fragments must not pass by word count.\n                if sum(bool(g) for g in groups) < 2 or not any(groups[:3]):\n                    continue\n                info = set.union(*groups)\n                score = (3 * len(groups[0]) + 6 * len(groups[1]) + 5 * len(groups[2])\n                         + 2 * len(groups[3]) + .5 * len(groups[4]) + 2 * len(info) / len(selected)\n                         - .7 * sum(w in stop for w in selected))\n                candidates.append({"caption": caption, "score": round(score, 8), "attributes": sorted(info)})\n                seen.add(caption)\n    candidates.sort(key=lambda x: (-x["score"], len(x["caption"].split()), x["caption"]))\n    best = candidates[0] if candidates else {"caption": "", "score": 0., "attributes": []}\n    return {**best, "candidate_count": len(candidates), "reason": "extractive_attributes" if candidates else "no_informative_span"}\n\ndef attribute_caption(response):\n    return attribute_caption_details(response)["caption"]\n\ndef prior_caption_registration(payload):\n    """Validate reusable long responses against original image/model hashes."""\n    import csv\n    import json\n    from pathlib import Path\n    reference = str(payload.get("prior_captions_source", "")).strip()\n    if not reference:\n        return None, {}\n    root = Path(reference).expanduser().resolve()\n    if root.is_file():\n        root = root.parent\n    candidates = [root, root / "shared/caption_generation"]\n    found = []\n    for folder in candidates:\n        csv_path = folder / "captions.csv"\n        for protocol_path, status_path in [\n            (folder / "generation_protocol/protocol.json", folder / "status.json"),\n            (folder / "caption_provenance/generation_protocol/protocol.json", folder / "caption_provenance/status.json"),\n        ]:\n            if csv_path.is_file() and protocol_path.is_file() and status_path.is_file():\n                found.append((csv_path, protocol_path, status_path))\n    if len(found) != 1:\n        raise FileNotFoundError("LONG_CAPTIONS_SOURCE must be the old bundle or shared/caption_generation, with CSV, generation protocol and status")\n    csv_path, protocol_path, status_path = found[0]\n    protocol, status = json.loads(protocol_path.read_text("utf-8")), json.loads(status_path.read_text("utf-8"))\n    if not status.get("complete") or file_sha256(csv_path) != status.get("csv_sha256") or object_hash(protocol) != status.get("identity"):\n        raise ValueError("Original caption CSV/protocol/status is incomplete or changed")\n    if protocol.get("model_assets") != checkpoint_asset_identity(payload["model_path"]):\n        raise ValueError("Reuse only long responses produced by this same frozen Florence checkpoint")\n    image_binding = {r["image_id"]: r for r in protocol["images"]}\n    if len(image_binding) != len(protocol["images"]):\n        raise ValueError("Duplicate original image IDs")\n    with csv_path.open(encoding="utf-8-sig", newline="") as handle:\n        rows = list(csv.DictReader(handle))\n    lookup = {r["image_id"]: r for r in rows}\n    if len(lookup) != len(rows):\n        raise ValueError("Duplicate caption rows")\n    reused = {}\n    for record in payload["records"]:\n        bound, row = image_binding.get(record["image_id"]), lookup.get(record["image_id"])\n        if bound is None or row is None:\n            continue  # Generate only genuinely missing images.\n        if (bound.get("image_sha256") != record["image_sha256"] or bound.get("split") != record["split"]\n                or row.get("split") != record["split"] or row.get("captioner_fingerprint") != status["identity"]):\n            raise ValueError("Original long caption belongs to another image/split: " + record["image_id"])\n        if not row.get("source_long_caption"):\n            continue\n        reused[record["image_id"]] = row\n    identity = {"csv_sha256": status["csv_sha256"], "generation_identity": status["identity"],\n                "reusable_images": len(reused), "operation": "reextract_original_response_not_human_ground_truth"}\n    return identity, reused\n\ndef generate_attribute_records(payload, identity, factory):\n    """Seed a NEW per-image cache; never change the old CSV or old checkpoint."""\n    from pathlib import Path\n    registration, source = prior_caption_registration(payload)\n    reused = 0\n    directory = Path(payload["output"]) / "generated"\n    for record in payload["records"]:\n        if record["image_id"] not in source or cached_row(directory, record, identity) is not None:\n            continue\n        original = source[record["image_id"]]\n        limited = str(original["generation_token_limit_reached"]).lower() == "true"\n        row = make_generation_row(record, original["source_long_caption"], limited, identity)\n        atomic_json(cache_path(directory, record["image_id"]), {"identity": identity, "row": row})\n        reused += 1\n    print("Re-extracted long responses in NEW cache:", reused, flush=True)\n    return generate_prefix_records(payload, identity, factory=factory)\n\ndef clean_dtd_caption(text):\n    return attribute_caption(text)\n\ndef generate_isic_records(payload, identity, factory=IsicFlorencePrefixCaptioner):\n    return generate_attribute_records(payload, identity, factory)\n\n\n\n\ndef validate_retry_settings(settings):\n    for key in ("batch_size", "max_new_tokens", "retry_greedy_tokens", "retry_beam_tokens", "retry_beams", "retry_returns"):\n        if type(settings.get(key)) is not int or settings[key] < 1:\n            raise ValueError(key + " must be a positive integer")\n    if settings["retry_beams"] < 2 or settings["retry_returns"] > settings["retry_beams"]:\n        raise ValueError("Require 2 <= retry_beams and retry_returns <= retry_beams")\n    if (settings.get("precision") != "float32" or type(settings.get("seed")) is not int\n            or not 0 <= settings["seed"] < 2**32):\n        raise ValueError("Use the frozen FP32 Florence protocol with an integer seed")\n\n\nclass IsicFlorenceFirstValidCaptioner(IsicFlorencePrefixCaptioner):\n    def generate_candidates(self, images, *, max_new_tokens, num_beams, num_return_sequences):\n        import torch\n        if num_return_sequences > num_beams or (num_beams == 1 and num_return_sequences != 1):\n            raise ValueError("Invalid deterministic beam/return configuration")\n        inputs = self.processor(text=[TASK_PROMPT] * len(images), images=images, return_tensors="pt", padding=True)\n        moved = {k:v.to(device=self.device, dtype=self.dtype if k == "pixel_values" else v.dtype)\n                 for k,v in inputs.items() if torch.is_tensor(v)}\n        with torch.inference_mode():\n            output = self.model.generate(**moved, max_new_tokens=max_new_tokens, num_beams=num_beams,\n                num_return_sequences=num_return_sequences, do_sample=False, use_cache=True, early_stopping=False)\n        decoded = self.processor.batch_decode(output, skip_special_tokens=False)\n        if len(decoded) != len(images) * num_return_sequences:\n            raise ValueError("Incomplete grouped Florence beam outputs")\n        sequences = output.cpu().tolist()\n        eos = self.model.generation_config.eos_token_id\n        eos = set(eos if isinstance(eos, (list,tuple)) else [eos])\n        pad = self.model.generation_config.pad_token_id\n        groups = []\n        for image_index, image in enumerate(images):\n            candidates = []\n            for rank in range(num_return_sequences):\n                index = image_index * num_return_sequences + rank\n                parsed = self.processor.post_process_generation(decoded[index], task=TASK_PROMPT,\n                    image_size=(image.width,image.height))\n                if not isinstance(parsed,dict) or not isinstance(parsed.get(TASK_PROMPT),str):\n                    raise ValueError("Florence returned another task/field")\n                # Exclude the initial decoder token (which may itself be EOS).\n                # Strip padding from shorter rows in a multi-image batch. EOS\n                # forced at the length cap does not certify a complete response.\n                new_tokens = list(sequences[index][1:])\n                if pad not in eos:\n                    while new_tokens and new_tokens[-1] == pad:\n                        new_tokens.pop()\n                candidates.append({"response":parsed[TASK_PROMPT].strip(),\n                    "token_limit":len(new_tokens) >= max_new_tokens, "new_tokens":len(new_tokens), "rank":rank})\n            groups.append(candidates)\n        return groups\n\n\ndef firstvalid_audit_path(output, image_id):\n    return Path(output) / "retry_audit" / (object_hash(image_id) + ".json")\n\n\ndef firstvalid_write(output, record, identity, trace):\n    atomic_json(firstvalid_audit_path(output,record["image_id"]), trace)\n    if trace["complete"]:\n        chosen = trace["selected_row"]\n        row = dict(chosen, baseline_v2_caption=trace["baseline_row"]["final_caption"],\n            baseline_v2_valid=bool(trace["baseline_row"]["caption_valid"]),\n            source_caption_valid=trace["source_caption_valid"],\n            selection_origin=trace["selection_origin"],\n            retry_count=sum(len(r["candidates"]) for r in trace["rounds"]),\n            retry_rounds=len(trace["rounds"]))\n        atomic_json(cache_path(Path(output)/"generated",record["image_id"]), {"identity":identity,"row":row})\n\n\ndef generate_firstvalid_records(payload, identity, factory=IsicFlorenceFirstValidCaptioner):\n    import gc\n    import torch\n    from PIL import Image, ImageOps\n    from tqdm.auto import tqdm\n    settings, records, output = payload["settings"], payload["records"], Path(payload["output"])\n    validate_retry_settings(settings)\n    registration, source = prior_caption_registration(payload)\n    if registration is None or len(source) != len(records):\n        raise ValueError("Attach complete registered v1/v2 long captions for every image. This experiment retries only v2 rejects.")\n    traces = {}\n    for record in records:\n        path = firstvalid_audit_path(output, record["image_id"])\n        if path.is_file():\n            trace = json.loads(path.read_text("utf-8"))\n            if trace.get("identity") != identity or trace.get("image_id") != record["image_id"]:\n                raise ValueError("Retry audit belongs to another image/protocol")\n            if trace["complete"]:\n                firstvalid_write(output,record,identity,trace)  # Repair an interrupted atomic commit.\n        else:\n            if cache_path(output/"generated",record["image_id"]).is_file():\n                raise ValueError("A generated row exists without its retry provenance; use a fresh output")\n            original = source[record["image_id"]]\n            baseline = make_generation_row(record, original["source_long_caption"],\n                str(original["generation_token_limit_reached"]).lower() == "true", identity)\n            trace = {"identity":identity,"image_id":record["image_id"],"baseline_row":baseline,\n                "source_caption_valid":str(original["caption_valid"]).lower() == "true",\n                "rounds":[],"selected_row":baseline,"selection_origin":"baseline_v2",\n                "complete":bool(baseline["caption_valid"])}\n            firstvalid_write(output,record,identity,trace)\n        traces[record["image_id"]] = trace\n    baseline_masked = sum(not t["baseline_row"]["caption_valid"] for t in traces.values())\n    unresolved = [r for r in records if not traces[r["image_id"]]["complete"]]\n    if len(unresolved) > baseline_masked:\n        raise ValueError("Unexpected retry population")\n    print(f"Images={len(records)}; baseline-v2 masked={baseline_masked}; remaining retry images={len(unresolved)}",flush=True)\n    captioner = None\n    schedule = [("greedy_retry",settings["retry_greedy_tokens"],1,1),\n                ("beam_retry",settings["retry_beam_tokens"],settings["retry_beams"],settings["retry_returns"])]\n    try:\n        for phase, tokens, beams, returns in schedule:\n            todo = [r for r in records if not traces[r["image_id"]]["complete"]\n                    and not any(x["phase"] == phase for x in traces[r["image_id"]]["rounds"])]\n            if not todo:\n                continue\n            if captioner is None:\n                captioner = factory(payload["model_path"],settings)\n            offset, batch_size = 0, settings["batch_size"]\n            progress = tqdm(total=len(todo),desc="ISIC " + phase,unit="image")\n            try:\n                while offset < len(todo):\n                    batch = todo[offset:offset + batch_size]\n                    images = []\n                    try:\n                        for record in batch:\n                            if file_sha256(record["image_path"]) != record["image_sha256"]:\n                                raise ValueError("Image changed before retry: " + record["image_id"])\n                            with Image.open(record["image_path"]) as im:\n                                images.append(ImageOps.exif_transpose(im).convert("RGB"))\n                        generated = captioner.generate_candidates(images,max_new_tokens=tokens,\n                            num_beams=beams,num_return_sequences=returns)\n                        if len(generated) != len(batch) or any(len(g) != returns for g in generated):\n                            raise ValueError("Incomplete retry response group")\n                    except torch.cuda.OutOfMemoryError:\n                        if batch_size == 1:\n                            raise RuntimeError("Florence retry OOM at batch 1; committed audit is resumable. Free GPU memory and rerun") from None\n                        batch_size = max(1,batch_size//2)\n                        gc.collect(); torch.cuda.empty_cache()\n                        continue\n                    finally:\n                        for im in images:\n                            im.close()\n                    for record, candidates in zip(batch,generated):\n                        trace = traces[record["image_id"]]\n                        attempts = []\n                        chosen = None\n                        for candidate in candidates:\n                            row = make_generation_row(record,candidate["response"],candidate["token_limit"],identity)\n                            attempts.append({"rank":candidate["rank"],"new_tokens":candidate["new_tokens"],"row":row})\n                            if chosen is None and row["caption_valid"]:\n                                chosen = row\n                        trace["rounds"].append({"phase":phase,"max_new_tokens":tokens,"num_beams":beams,\n                            "num_return_sequences":returns,"candidates":attempts})\n                        if chosen is not None:\n                            trace.update(selected_row=chosen,selection_origin=phase,complete=True)\n                        elif phase == "beam_retry":\n                            trace.update(complete=True,selection_origin="still_masked_after_retries")\n                        firstvalid_write(output,record,identity,trace)\n                    offset += len(batch); progress.update(len(batch))\n            finally:\n                progress.close()\n    finally:\n        del captioner\n        gc.collect()\n        torch.cuda.empty_cache()\n    if not all(t["complete"] for t in traces.values()):\n        raise RuntimeError("Retries are incomplete; rerun to resume")\n    rows = [cached_row(output/"generated",r,identity) for r in records]\n    if any(r is None for r in rows):\n        raise ValueError("Missing committed final caption")\n    valid = sum(bool(r["caption_valid"]) for r in rows)\n    recovered = sum(not t["baseline_row"]["caption_valid"] and t["selected_row"]["caption_valid"] for t in traces.values())\n    if len(records)-valid != baseline_masked-recovered:\n        raise ValueError("A valid baseline caption was lost or mask counts are inconsistent")\n    columns = COLUMNS + ["baseline_v2_caption","baseline_v2_valid","source_caption_valid","selection_origin","retry_rounds"]\n    atomic_csv(output/"captions.csv",rows,columns)\n    status = {"complete":True,"rows":len(rows),"expected":len(records),"identity":identity,\n        "csv_sha256":file_sha256(output/"captions.csv"),"csv_path":str((output/"captions.csv").resolve()),\n        "baseline_v2_masked":baseline_masked,"recovered_from_v2":recovered,"final_masked":len(records)-valid,\n        "valid_by_split":{role:sum(bool(r["caption_valid"]) for r in rows if r["split"]==role) for role in ("train","val","test")}}\n    atomic_json(output/"status.json",status)\n    print(json.dumps(status,indent=2),flush=True)\n    return status\n\n\ndef firstvalid_worker_main(request_path, phase):\n    import torch\n    from importlib.metadata import version\n    payload = json.loads(Path(request_path).read_text("utf-8"))\n    validate_caption_records(payload["records"])\n    validate_retry_settings(payload["settings"])\n    if not torch.cuda.is_available():\n        raise RuntimeError("Enable a Kaggle GPU")\n    if not version("transformers").startswith("4.49."):\n        raise RuntimeError("Use the isolated offline Transformers 4.49 worker")\n    torch.set_float32_matmul_precision("highest")\n    torch.backends.cuda.matmul.allow_tf32=False\n    torch.backends.cudnn.allow_tf32=False\n    torch.manual_seed(payload["settings"]["seed"])\n    torch.cuda.manual_seed_all(payload["settings"]["seed"])\n    if phase == "generate":\n        path,count = validate_florence_prefix_folder(payload["model_path"])\n        registration,_ = prior_caption_registration(payload)\n        if registration is None:\n            raise ValueError("A registered old caption source is required")\n        protocol = {"schema":1,"method":METHOD,"model_id":MODEL_ID,"policy":POLICY,"prompt":TASK_PROMPT,\n            "model_assets":checkpoint_asset_identity(path),"parameters":count,"dtype":"float32",\n            "decoding":"keep_valid_v2_then_greedy_retry_then_first_valid_beam",\n            "long_response_source":registration,"settings":payload["settings"],"runtime":runtime_versions(),\n            "code_sha256":payload["code_sha256"],\n            "images":[{k:r[k] for k in ("image_id","split","image_sha256")} for r in payload["records"]]}\n        identity = lock_protocol(Path(payload["output"])/"generation_protocol/protocol.json",protocol)\n        return generate_firstvalid_records(payload,identity)\n    if phase == "encode":\n        return encode_isic_texts(payload)\n    raise ValueError("Unknown caption-only worker phase")\n\n\ndef resolve_prior_source(reference, roots):\n    """Find one original Florence run (duplicate copies are grouped by CSV hash)."""\n    explicit = str(reference).strip()\n    paths = [Path(explicit).expanduser()] if explicit else []\n    searches = paths or [Path(r) for r in roots]\n    if explicit and not paths[0].exists():\n        raise FileNotFoundError("PRIOR_CAPTIONS_SOURCE does not exist: " + explicit)\n    discovered = []\n    for root in searches:\n        if root.is_file():root=root.parent\n        if not root.is_dir():continue\n        for current,dirs,files in os.walk(root):\n            folder=Path(current)\n            dirs[:] = [n for n in dirs if n not in {"images","masks","candidate_images","ground_truth_masks","caption_models","generated","caption_runtime","vendor","legacy_packages",".cache","__pycache__"}]\n            if len(folder.relative_to(root).parts)>=10:dirs[:]=[]\n            if "captions.csv" not in files:continue\n            for proto,status in [(folder/"generation_protocol/protocol.json",folder/"status.json"),\n                    (folder/"caption_provenance/generation_protocol/protocol.json",folder/"caption_provenance/status.json")]:\n                if not proto.is_file() or not status.is_file():continue\n                p=json.loads(proto.read_text("utf-8"));s=json.loads(status.read_text("utf-8"))\n                if p.get("method") not in {"florence_prefix_control","florence_extractive_attributes_v2"}:continue\n                if not s.get("complete") or file_sha256(folder/"captions.csv")!=s.get("csv_sha256") or object_hash(p)!=s.get("identity"):\n                    raise ValueError("Source caption CSV/protocol/status changed: " + str(folder))\n                discovered.append((folder.resolve(),p,s))\n    by_hash={}\n    for item in discovered:by_hash.setdefault(item[2]["csv_sha256"],[]).append(item)\n    if len(by_hash)!=1:\n        raise FileNotFoundError("Attach the registered v1 or v2 ISIC Florence output/bundle and set PRIOR_CAPTIONS_SOURCE exactly. "\n            f"Found {len(by_hash)} different caption sources: {[str(x[0]) for x in discovered]}")\n    folder,protocol,status=min(next(iter(by_hash.values())),key=lambda x:(len(x[0].parts),str(x[0])))\n    return folder,protocol,status\n\n\ndef recover_image_records(protocol, dataset_root, roots):\n    """Read RGB image files only. Segmentation masks never enter this pipeline."""\n    from tqdm.auto import tqdm\n    records=protocol.get("images",[])\n    if not records or len({r["image_id"] for r in records})!=len(records):\n        raise ValueError("Source image binding is empty or duplicated")\n    wanted={r["image_id"] for r in records}\n    for r in records:\n        if (set(r)!={"image_id","split","image_sha256"} or not re.fullmatch(r"ISIC_\\d+",r["image_id"])\n                or r["split"] not in {"train","val","test"} or not re.fullmatch(r"[a-f0-9]{64}",r["image_sha256"])):\n            raise ValueError("Malformed original pixel-only image registration")\n    searches=[Path(dataset_root).expanduser()] if str(dataset_root).strip() else [Path(r) for r in roots]\n    if str(dataset_root).strip() and not searches[0].is_dir():raise FileNotFoundError("ISIC_ROOT not found")\n    index={}\n    for root in searches:\n        if not root.is_dir():continue\n        for current,dirs,files in os.walk(root):\n            dirs[:]=[n for n in dirs if n not in {".git",".cache","__pycache__","models","caption_models","candidate_images","ground_truth_masks","masks","generated","vendor","legacy_packages"}\n                     and "groundtruth" not in n.lower() and "ground_truth" not in n.lower()]\n            for name in files:\n                match=re.fullmatch(r"(ISIC_\\d+)\\.(?:jpg|jpeg|png|bmp|webp)",name,re.I)\n                if match and match[1].upper() in wanted:\n                    index.setdefault(match[1].upper(),[]).append(Path(current)/name)\n    result=[]\n    digests={}\n    for row in tqdm(records,desc="Verify original RGB image hashes"):\n        path=None\n        for candidate in sorted(set(p.resolve() for p in index.get(row["image_id"],[]))):\n            if candidate not in digests:\n                digests[candidate]=file_sha256(candidate)\n            digest=digests[candidate]\n            if digest==row["image_sha256"]:path=candidate;break\n        if path is None:raise FileNotFoundError("No original RGB image with matching hash for " + row["image_id"] + "; attach ISIC and set ISIC_ROOT")\n        result.append(dict(row,image_path=str(path)))\n    validate_caption_records(result)\n    return result\n\n\ndef write_firstvalid_reports(output):\n    import csv\n    import collections\n    output=Path(output)\n    with (output/"captions.csv").open(encoding="utf-8-sig",newline="") as handle:rows=list(csv.DictReader(handle))\n    true=lambda x:str(x).lower()=="true"\n    summaries=[]\n    for role in ("all","train","val","test"):\n        subset=rows if role=="all" else [r for r in rows if r["split"]==role]\n        if not subset:continue\n        before=sum(not true(r["baseline_v2_valid"]) for r in subset)\n        after=sum(not true(r["caption_valid"]) for r in subset)\n        recovered=sum(not true(r["baseline_v2_valid"]) and true(r["caption_valid"]) for r in subset)\n        if after!=before-recovered:raise ValueError("Mask comparison is inconsistent")\n        if any(true(r["baseline_v2_valid"]) and (not true(r["caption_valid"]) or r["baseline_v2_caption"]!=r["final_caption"]) for r in subset):\n            raise ValueError("A valid v2 caption was changed")\n        summaries.append({"split":role,"images":len(subset),"source_csv_masked":sum(not true(r["source_caption_valid"]) for r in subset),\n            "baseline_v2_masked":before,"firstvalid_masked":after,"recovered_from_v2":recovered,\n            "baseline_mask_percent":100*before/len(subset),"firstvalid_mask_percent":100*after/len(subset)})\n    atomic_csv(output/"mask_comparison.csv",summaries,list(summaries[0]))\n    counts=collections.Counter(r["final_caption"] for r in rows if true(r["caption_valid"]))\n    review=[]\n    for row in rows:\n        trace=json.loads(firstvalid_audit_path(output,row["image_id"]).read_text("utf-8"))\n        review.append({"image_id":row["image_id"],"split":row["split"],"baseline_caption":row["baseline_v2_caption"],\n            "baseline_valid":row["baseline_v2_valid"],"final_caption":row["final_caption"],"final_valid":row["caption_valid"],\n            "selection_origin":row["selection_origin"],"baseline_long_caption":trace["baseline_row"]["source_long_caption"],\n            "selected_long_caption":row["source_long_caption"],"retry_count":row["retry_count"]})\n    atomic_csv(output/"caption_retries_review.csv",review,list(review[0]))\n    report={"policy":POLICY,"method":METHOD,"mask_comparison":summaries,"unique_valid_captions":len(counts),\n        "top_valid_captions":counts.most_common(10),"selection_origins":dict(collections.Counter(r["selection_origin"] for r in rows)),\n        "filters_relaxed":False,"no_gt_masks_or_diagnoses_in_generation_inputs":True,\n        "segmentation_training_performed":False,"not_caption_factual_accuracy_or_segmentation_metrics":True,\n        "manual_image_review_required":True}\n    atomic_json(output/"caption_quality_report.json",report)\n    return report\n\nif __name__ == "__main__":\n    firstvalid_worker_main(sys.argv[1], sys.argv[2])\n', encoding="utf-8")
spec = importlib.util.spec_from_file_location("isic_caption_firstvalid_runtime", RUNTIME_FILE)
core = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = core
spec.loader.exec_module(core)
core.validate_retry_settings(SETTINGS)
print("Method:", core.METHOD, "| unchanged policy:", core.POLICY)


In [ ]:
# Read only registered pixel/split metadata. No segmentation masks are opened.
SOURCE_DIR, SOURCE_PROTOCOL, SOURCE_STATUS = core.resolve_prior_source(PRIOR_CAPTIONS_SOURCE, SEARCH_ROOTS)
if OUTPUT_DIR == SOURCE_DIR or OUTPUT_DIR in SOURCE_DIR.parents or SOURCE_DIR in OUTPUT_DIR.parents:
    raise ValueError("Use a separate output folder outside the old source; do not overwrite a run/bundle")
CAPTION_RECORDS = core.recover_image_records(SOURCE_PROTOCOL, ISIC_ROOT, SEARCH_ROOTS)
core.atomic_json(OUTPUT_DIR / "resolved_image_records.json", CAPTION_RECORDS)
from collections import Counter
print("Old caption source:", SOURCE_DIR)
print("Exact saved split:", dict(Counter(r["split"] for r in CAPTION_RECORDS)))
print("Original CSV SHA256:", SOURCE_STATUS["csv_sha256"])


In [ ]:
# Use the Florence weights/wheels already exported with the old bundle when present.
model_ref = FLORENCE_MODEL_REF
if not model_ref and (SOURCE_DIR / "caption_models/florence").is_dir():
    model_ref = str(SOURCE_DIR / "caption_models/florence")
if not model_ref:
    attached = Path("/kaggle/input/notebooks/nguyenducxu/notebook-mt-moi/Florence-2-base")
    model_ref = str(attached) if attached.is_dir() else core.MODEL_ID
MODEL_PATH, BACKEND = core.florence_checkpoint(model_ref, [SOURCE_DIR, *SEARCH_ROOTS])
if BACKEND != "legacy":
    raise ValueError("Use the original Microsoft Florence-2-base checkpoint with offline Transformers 4.49")
core.validate_florence_prefix_folder(MODEL_PATH)
PAYLOAD = {
    "output": str(OUTPUT_DIR), "model_path": str(MODEL_PATH), "settings": SETTINGS,
    "records": CAPTION_RECORDS, "prior_captions_source": str(SOURCE_DIR),
    "code_sha256": core.file_sha256(RUNTIME_FILE),
}
registration, old_captions = core.prior_caption_registration(PAYLOAD)
if len(old_captions) != len(CAPTION_RECORDS):
    raise ValueError("The source must contain registered long captions for every image")
asset_root = Path(OFFLINE_ASSETS_ROOT)
if not asset_root.is_dir():
    asset_root = SOURCE_DIR
CAPTION_ENV = core.caption_runtime_environment("legacy", asset_root, OUTPUT_DIR / "caption_runtime")
if ENCODE_CLIP_TEXT:
    clip_ref = CLIP_MODEL_LOCAL_PATH
    if not clip_ref and (SOURCE_DIR / "caption_models/clip").is_dir():
        clip_ref = str(SOURCE_DIR / "caption_models/clip")
    PAYLOAD["clip_path"] = str(core.resolve_clip_model(clip_ref, [SOURCE_DIR, *SEARCH_ROOTS]))
REQUEST_FILE = OUTPUT_DIR / "request.json"
core.atomic_json(REQUEST_FILE, PAYLOAD)
print("Frozen Florence:", MODEL_PATH)
print("Reusable long captions:", len(old_captions))


In [ ]:
# The child process uses isolated offline Transformers 4.49. The kernel is unchanged.
# Resume uses original image hashes, decoding settings and per-image retry audits.
core.run_worker(RUNTIME_FILE, REQUEST_FILE, "generate", CAPTION_ENV, OUTPUT_DIR / "generate.log")
NEW_CAPTIONS, CAPTION_STATUS = core.validated_export(PAYLOAD)
print("Complete CSV:", CAPTION_STATUS["csv_path"])
print("Baseline v2 masked:", CAPTION_STATUS["baseline_v2_masked"])
print("Recovered via retries:", CAPTION_STATUS["recovered_from_v2"])
print("Final masked:", CAPTION_STATUS["final_masked"])


In [ ]:
import pandas as pd
from IPython.display import display
QUALITY_REPORT = core.write_firstvalid_reports(OUTPUT_DIR)
display(pd.read_csv(OUTPUT_DIR / "mask_comparison.csv"))
review = pd.read_csv(OUTPUT_DIR / "caption_retries_review.csv").fillna("")
recovered = review[review["final_valid"].astype(str).str.lower().eq("true")
                   & review["baseline_valid"].astype(str).str.lower().eq("false")]
print("Sample of recovered captions (check these against their images):")
display(recovered.head(12))
print("The clinical blacklist, 5–8 word rule and attribute checks were NOT relaxed.")
print("Fewer masks do not establish factual caption accuracy or higher segmentation Dice.")


In [ ]:
# Visual review of a fixed small sample, never used to pick hyperparameters from test metrics.
from PIL import Image, ImageOps
from IPython.display import Markdown
record_by_id = {r["image_id"]:r for r in CAPTION_RECORDS}
review_sample = recovered.head(8) if len(recovered) else review.head(4)
for row in review_sample.to_dict("records"):
    display(Markdown("**" + row["image_id"] + "** (" + row["split"] + ")"))
    with Image.open(record_by_id[row["image_id"]]["image_path"]) as raw:
        thumbnail = ImageOps.exif_transpose(raw).convert("RGB")
        thumbnail.thumbnail((360, 260))
        display(thumbnail)
    print("V2:", row["baseline_caption"], "| valid:", row["baseline_valid"])
    print("New:", row["final_caption"], "| valid:", row["final_valid"], "|", row["selection_origin"])


In [ ]:
if ENCODE_CLIP_TEXT:
    core.run_worker(RUNTIME_FILE, REQUEST_FILE, "encode", CAPTION_ENV, OUTPUT_DIR / "encode.log")
    print("Optional NEW CLIP cache:", OUTPUT_DIR / "text_cache/clip_text.pt")
else:
    print("Skipped CLIP encoding: captions.csv is ready; no CLIP weights required in CSV-only mode.")


In [ ]:
import zipfile
from IPython.display import FileLink, display
core.atomic_json(OUTPUT_DIR / "experiment_settings.json", {
    "method": core.METHOD, "policy": core.POLICY, "settings": SETTINGS,
    "caption_csv_sha256": CAPTION_STATUS["csv_sha256"],
    "caption_only": True, "segmentation_training": False, "filters_relaxed": False,
    "prior_csv_sha256": SOURCE_STATUS["csv_sha256"],
    "clip_text_encoded": ENCODE_CLIP_TEXT,
    "runtime_note": "Future training/inference must use this retry/extraction policy and new embeddings, not an old checkpoint as a retrained result.",
})
# Small archive: captions, audit, decoding/runtime provenance. No model weights or RGB dataset.
archive = OUTPUT_DIR / "isic2017_captions_firstvalid_v3.zip"
top_files = ["captions.csv", "status.json", "mask_comparison.csv", "caption_retries_review.csv",
             "caption_quality_report.json", "experiment_settings.json", "caption_firstvalid_runtime.py"]
selected = [OUTPUT_DIR / name for name in top_files]
selected += sorted((OUTPUT_DIR / "generation_protocol").rglob("*.json"))
selected += sorted((OUTPUT_DIR / "retry_audit").rglob("*.json"))
if ENCODE_CLIP_TEXT:
    selected += sorted(p for p in (OUTPUT_DIR / "text_cache").rglob("*") if p.is_file())
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as handle:
    for path in selected:
        handle.write(path, path.relative_to(OUTPUT_DIR).as_posix())
print("Download captions.csv for the next M-TACS experiment; keep the ZIP for provenance.")
for name in ("captions.csv", "mask_comparison.csv", "caption_retries_review.csv", archive.name):
    display(FileLink(str(OUTPUT_DIR / name)))
print("Done. No segmentation model was trained or modified.")
